# Lección 2.3 — Preparación: escalado y codificación de categorías

> **Módulo 2 · Los datos: el combustible** · Nivel 1 — Cero
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 65 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lecciones 2.1 (tipos de datos) y 2.2 (calidad de datos)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar por qué algunos modelos necesitan que las columnas estén en la **misma escala** y otros no.
- Elegir entre `StandardScaler`, `MinMaxScaler` y `RobustScaler`.
- Convertir categorías en números con **ordinal**, **one-hot** y **target encoding**, sabiendo cuándo usar cada una.
- Combinar todo en un `ColumnTransformer`, la pieza que prepara cada columna según su tipo.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import (StandardScaler, MinMaxScaler, RobustScaler,
                                   OneHotEncoder, OrdinalEncoder, TargetEncoder)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_wine

plt.rcParams["figure.figsize"] = (8, 4)
print("✅ Todo listo")

---
## 1. 📖 La intuición: comparar pesos con dólares

Imagina que comparas dos departamentos:
- A: renta de **20,000** pesos y **2** habitaciones.
- B: renta de **20,500** pesos y **5** habitaciones.

¿Se parecen? Si sumas las diferencias "a lo bruto", la diferencia de renta (500) aplasta a la de habitaciones (3), y concluirías que son casi iguales. Pero tener 2 o 5 habitaciones es una diferencia **enorme**.

El problema es que cada columna está en su **propia unidad** (pesos, habitaciones, años, kilos…). Es como comparar precios en pesos con precios en dólares sin convertir. **Preparar los datos** es poner todas las columnas en un "idioma común" de números que el modelo pueda comparar de forma justa.

Esto incluye dos tareas:
1. **Escalar** las columnas numéricas para que ninguna "grite" más fuerte que las demás.
2. **Codificar** las categorías ("Roma", "Polanco") como números, sin inventar información falsa.

### ✋ Ejercicio 1 — Calentamiento
Una app de citas quiere encontrar personas "parecidas" usando **edad** (18–70 años) y **ingreso anual** (50,000–2,000,000 pesos). Si mide la similitud sumando diferencias sin preparar nada, ¿qué columna decidirá casi todo? ¿Te parece bien?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Decidiría el **ingreso**, porque sus diferencias son de cientos de miles, mientras que las de edad son de unas decenas. La app emparejaría a una persona de 20 años con una de 65 solo porque ganan parecido. No es que el ingreso sea más importante: simplemente **está escrito con números más grandes**.
</details>

---
## 2. 📖 Escalado: que ninguna columna grite más fuerte

Algunos modelos calculan **distancias** entre ejemplos (como KNN, "dime con quién andas", Lección 5.3). Para ellos, la escala lo es todo.

### 🔮 Ejercicio 2 — ¿Quién se parece más a Ana?
Ana tiene 25 años y gana 30,000 al mes. Antes de ejecutar: ¿quién se parece más a ella, **Beto** (60 años, gana 31,000) o **Caro** (26 años, gana 34,000)? ¿Y qué dirá la computadora si mide la distancia sin escalar?

✍️ *Mi apuesta:* Para mí: ___ · Para la computadora: ___

In [ ]:
personas = pd.DataFrame({"edad": [25, 60, 26, 45, 33, 52],
                         "ingreso": [30_000, 31_000, 34_000, 20_000, 45_000, 60_000]},
                        index=["Ana", "Beto", "Caro", "Dani", "Eli", "Fede"])

def distancia(tabla, a, b):
    return np.sqrt(((tabla.loc[a] - tabla.loc[b]) ** 2).sum())   # distancia "en línea recta"

print("SIN escalar")
print(f"  Ana ↔ Beto: {distancia(personas, 'Ana', 'Beto'):>8.1f}")
print(f"  Ana ↔ Caro: {distancia(personas, 'Ana', 'Caro'):>8.1f}")

escalada = pd.DataFrame(StandardScaler().fit_transform(personas),
                        columns=personas.columns, index=personas.index)
print("\nCON escalado estándar")
print(f"  Ana ↔ Beto: {distancia(escalada, 'Ana', 'Beto'):>8.2f}")
print(f"  Ana ↔ Caro: {distancia(escalada, 'Ana', 'Caro'):>8.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **Sin escalar**, la computadora dice que **Beto** es el más parecido (≈1,000 contra ≈4,000), porque 1,000 pesos de diferencia "pesan" mucho más que 35 años. Absurdo.
- **Con escalado**, gana **Caro** (≈0.32 contra ≈2.66): casi la misma edad y un ingreso parecido.

El escalado no cambió a las personas: cambió la **forma de medirlas** para que cada columna cuente de manera justa.
</details>

### 📖 Tres formas de escalar
| Escalador | Qué hace | Resultado típico | Úsalo cuando… |
|---|---|---|---|
| `StandardScaler` (estandarizar) | Resta la media y divide entre la desviación estándar | Media 0, la mayoría entre −2 y 2 | Es el punto de partida habitual |
| `MinMaxScaler` (normalizar) | Lleva el mínimo a 0 y el máximo a 1 | Todo entre 0 y 1 | Necesitas un rango fijo (p. ej. píxeles de una imagen) |
| `RobustScaler` | Usa la **mediana** y el rango intercuartílico (Lección 2.2) | Centrado en 0, sin dejarse arrastrar por extremos | Hay **atípicos** que no puedes quitar |

### 🔮 Ejercicio 3 — El atípico aplastador
Estos son los minutos que tardan 6 repartidores en entregar un pedido. Uno se perdió y tardó 100 minutos. Antes de ejecutar: con `MinMaxScaler`, ¿dónde quedarán los 5 repartidores normales: repartidos entre 0 y 1, o todos amontonados cerca de algún valor?

In [ ]:
minutos = pd.DataFrame({"minutos": [10, 12, 11, 13, 12, 100]})

comparacion = pd.DataFrame({
    "original": minutos["minutos"],
    "Standard": StandardScaler().fit_transform(minutos).ravel(),
    "MinMax":   MinMaxScaler().fit_transform(minutos).ravel(),
    "Robust":   RobustScaler().fit_transform(minutos).ravel(),
}).round(2)
comparacion

<details><summary><b>👀 Ver respuesta</b></summary>

Con `MinMaxScaler`, los 5 normales quedan **amontonados entre 0 y 0.03**, y el perdido en 1. Toda la diferencia entre "rápido" (10 min) y "lento" (13 min) se aplasta. Con `StandardScaler` pasa algo parecido (todos ≈ −0.5). `RobustScaler` es el único que conserva las diferencias entre los normales (de −1.33 a 0.67) y deja al atípico muy lejos (≈59).

**Moraleja:** si hay atípicos que no puedes eliminar, usa `RobustScaler`.
</details>

### 📖 ¿Todos los modelos necesitan escalado? No.
- **Sí lo necesitan** los que miden distancias o suman columnas con pesos: KNN, SVM, regresión lineal/logística con regularización, redes neuronales.
- **No lo necesitan** los **árboles** y sus ensambles (Random Forest, *boosting*): cada pregunta mira **una sola columna** ("¿ingreso > 20,000?"), así que la escala no importa.

### 🔮 Ejercicio 4 — Vinos con y sin escalado
El conjunto `wine` tiene 178 vinos de 3 viñedos, con 13 mediciones químicas en escalas muy distintas (el alcohol vale ~13, la prolina ~750). Vamos a medir la exactitud de un **KNN** y de un **árbol**, con y sin escalado. Antes de ejecutar: ¿cuál de los dos modelos cambiará más al escalar?

✍️ *Mi apuesta:*

In [ ]:
X_vino, y_vino = load_wine(return_X_y=True, as_frame=True)
print(X_vino.describe().loc[["min", "max"]].round(1).iloc[:, [0, 4, 12]], "\n")

modelos = {
    "KNN sin escalar ": KNeighborsClassifier(),
    "KNN escalado    ": make_pipeline(StandardScaler(), KNeighborsClassifier()),
    "Árbol sin escalar": DecisionTreeClassifier(random_state=0),
    "Árbol escalado   ": make_pipeline(StandardScaler(), DecisionTreeClassifier(random_state=0)),
}
for nombre, modelo in modelos.items():
    exactitud = cross_val_score(modelo, X_vino, y_vino, cv=5).mean()
    print(f"{nombre}: {exactitud:.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

El **KNN** pasa de ≈69 % a ≈95 %: un salto enorme, sin cambiar nada del modelo. Sin escalar, la distancia la decidía casi solo la prolina (valores de cientos). El **árbol** da **exactamente lo mismo** con y sin escalado: sus preguntas comparan una columna a la vez, y escalar solo mueve el número del corte.

> 📝 `make_pipeline(StandardScaler(), KNeighborsClassifier())` crea una **tubería** (*pipeline*): primero escala, luego clasifica, como una sola pieza. En la Lección 2.6 verás por qué esto es mucho más que una comodidad.
</details>

---
## 3. 📖 Codificación ordinal: categorías con orden

Si una categoría tiene **orden natural** (Lección 2.1), puedes numerarla respetándolo: Básico = 0, Estándar = 1, Premium = 2. El modelo podrá usar ese orden ("a mayor plan, mayor…").

⚠️ El orden lo das **tú**. Si dejas que la computadora lo elija, usará el orden **alfabético**, que casi nunca es el correcto.

### 🔮 Ejercicio 5 — El orden alfabético
Antes de ejecutar: con el orden alfabético, ¿qué número le tocará a "Premium"? ¿Tiene sentido?

In [ ]:
planes = pd.DataFrame({"plan": ["Básico", "Premium", "Estándar", "Básico", "Premium"]})

alfabetico = OrdinalEncoder().fit(planes)
correcto = OrdinalEncoder(categories=[["Básico", "Estándar", "Premium"]]).fit(planes)  # ✏️ el orden lo das tú

planes["alfabetico"] = alfabetico.transform(planes[["plan"]]).ravel()
planes["correcto"] = correcto.transform(planes[["plan"]]).ravel()
planes

<details><summary><b>👀 Ver respuesta</b></summary>

Con el orden alfabético (Básico, Estándar, Premium) **coincide por suerte**: B < E < P. ¡Pero solo por suerte! Si los planes se llamaran "Bronce", "Plata" y "Oro", el orden alfabético sería Bronce = 0, **Oro = 1, Plata = 2**: el Oro quedaría por debajo de la Plata. **Declara siempre el orden explícitamente**, como en `correcto`.
</details>

---
## 4. 📖 One-hot: una columna de "sí/no" por categoría

Para categorías **sin orden** (ciudad, color, método de pago), numerarlas 1, 2, 3 inventa un orden falso (recuerda el "color promedio" verde de la Lección 2.1).

La solución clásica es **one-hot** (*uno caliente*): se crea **una columna por categoría**, que vale 1 si el ejemplo pertenece a ella y 0 si no. Cada fila tiene un solo "1" encendido: de ahí el nombre.

```
ciudad          ciudad_CDMX  ciudad_GDL  ciudad_MTY
CDMX       →        1            0           0
MTY        →        0            0           1
GDL        →        0            1           0
```

### 🔮 Ejercicio 6 — ¿Cuántas columnas?
Tenemos 6 clientes de 4 ciudades distintas y 2 métodos de pago. Antes de ejecutar: ¿cuántas columnas tendrá la tabla después del one-hot?

✍️ *Mi apuesta:*

In [ ]:
clientes = pd.DataFrame({
    "ciudad": ["CDMX", "MTY", "GDL", "CDMX", "Puebla", "GDL"],
    "pago":   ["tarjeta", "efectivo", "tarjeta", "tarjeta", "efectivo", "efectivo"],
})
onehot = OneHotEncoder(sparse_output=False).fit(clientes)
pd.DataFrame(onehot.transform(clientes), columns=onehot.get_feature_names_out()).astype(int)

<details><summary><b>👀 Ver respuesta</b></summary>

**6 columnas**: 4 de ciudad + 2 de pago. Ningún número "es mayor" que otro: cada categoría tiene su propio interruptor.

> 💡 Con `drop="if_binary"` puedes quedarte con **una** sola columna para las categorías de dos valores (pago = tarjeta: 1/0), porque la otra no aporta nada nuevo.
</details>

### 📖 ¿Y si llega una ciudad nueva?
El modelo se entrena hoy con CDMX, GDL, MTY y Puebla. Mañana llega un cliente de **Mérida**. ¿Qué pasa?

In [ ]:
nuevo = pd.DataFrame({"ciudad": ["Mérida"], "pago": ["tarjeta"]})
try:
    onehot.transform(nuevo)
except ValueError as error:
    print("❌ Error:", str(error)[:90], "...")

tolerante = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit(clientes)   # ✏️
print("\n✅ Con handle_unknown='ignore':")
pd.DataFrame(tolerante.transform(nuevo), columns=tolerante.get_feature_names_out()).astype(int)

### ✋ Ejercicio 7
1. ¿Qué hace el codificador por defecto ante una categoría que nunca vio?
2. ¿Cómo queda codificado el cliente de Mérida con `handle_unknown="ignore"`? ¿Por qué es razonable?

✍️ *Tus respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Se detiene con un error.** En producción eso significa que el sistema se cae con el primer cliente de una ciudad nueva.
2. Todas las columnas de ciudad en **0** ("no es ninguna de las que conozco") y la de tarjeta en 1. Es razonable: el modelo usa el resto de la información y no se rompe. Por eso `handle_unknown="ignore"` es casi siempre el valor recomendado.
</details>

### 📖 El problema de las muchas categorías
One-hot funciona de maravilla con pocas categorías. Pero ¿y una columna con **3,000 códigos postales**? Crearía 3,000 columnas, casi todas en 0, y muchas categorías tendrían solo uno o dos clientes: el modelo no puede aprender nada fiable de ellas (y verás en la Lección 2.8 por qué tantas columnas son un problema).

Opciones:
- Agrupar las categorías raras en "Otras" (`OneHotEncoder(min_frequency=...)`).
- Usar **target encoding**, que veremos ahora.

---
## 5. 📖 Target encoding: reemplazar la categoría por lo que "suele pasar" en ella

La idea: en vez de crear muchas columnas, reemplaza cada categoría por **el promedio de la etiqueta** (*target*) en esa categoría.

Ejemplo: queremos predecir si un cliente **cancela** su suscripción. Si en la colonia "Roma" cancelaron 3 de cada 10 clientes, "Roma" se convierte en **0.30**. Una sola columna, sin importar cuántas colonias haya.

### 🔮 Ejercicio 8 — La colonia de un solo cliente
Calculemos el target encoding "a mano". Antes de ejecutar: la colonia "Juárez" tiene **un solo** cliente, y canceló. ¿Qué número le tocará? ¿Te fiarías de él?

In [ ]:
cancelaciones = pd.DataFrame({
    "colonia": ["Roma"] * 10 + ["Centro"] * 8 + ["Juárez"],
    "cancelo": [1, 0, 0, 1, 0, 0, 1, 0, 0, 0] + [0, 1, 0, 0, 0, 0, 1, 0] + [1],
})
resumen = cancelaciones.groupby("colonia")["cancelo"].agg(clientes="count", tasa_cancelacion="mean")
resumen

<details><summary><b>👀 Ver respuesta</b></summary>

"Juárez" recibe **1.0**: "el 100 % cancela". Con un solo cliente eso es **pura casualidad**, pero el modelo lo creería a ciegas: estaría **memorizando** a ese cliente (sobreajuste) y, peor aún, usando su propia respuesta como característica (¡una **fuga de datos**, Lección 2.6!).

Por eso el target encoding real tiene dos protecciones:
1. **Suavizado:** las categorías con pocos ejemplos se "acercan" al promedio general (≈0.32) en lugar de creerse su propio promedio.
2. **Ajuste cruzado:** al entrenar, cada fila recibe un valor calculado **sin usar su propia etiqueta**.

El `TargetEncoder` de scikit-learn hace las dos cosas automáticamente.
</details>

In [ ]:
codificador = TargetEncoder(target_type="binary", smooth=5, random_state=0)   # ✏️ prueba smooth=0.5 o smooth=20
codificador.fit(cancelaciones[["colonia"]], cancelaciones["cancelo"])

print(f"Promedio general de cancelación: {cancelaciones['cancelo'].mean():.2f}")
for colonia, valor in zip(codificador.categories_[0], codificador.encodings_[0]):
    print(f"  {colonia:<7} → {valor:.2f}")

### ✋ Ejercicio 9
Compara estos valores con la tabla "a mano" de arriba. ¿Qué le pasó a Juárez? ¿Y a Roma, que tiene más clientes?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**Juárez** bajó de 1.00 a **0.43**, mucho más cerca del promedio general (0.32): con un solo cliente, el codificador "no le cree" del todo a su promedio. **Roma** (10 clientes) casi no se movió (0.30 → 0.31) y **Centro** poco (0.25 → 0.28): con más ejemplos, el codificador confía más en el promedio de la propia colonia.

✏️ Con `smooth=20`, Juárez baja hasta 0.35 (casi el promedio general); con `smooth=0.5` se queda en 0.77 (casi se cree su propio promedio).

> 📝 Si no pones `smooth`, scikit-learn usa `"auto"`, que calcula el suavizado a partir de la variabilidad dentro de cada categoría. Con categorías de **un solo** ejemplo ese cálculo no funciona bien (¡Juárez se quedaría en 1.00!), por eso aquí fijamos un número.
</details>

---
## 6. 📖 Todo junto: `ColumnTransformer`

En una tabla real tienes columnas de varios tipos, y **cada tipo necesita su propio tratamiento**. `ColumnTransformer` es como una línea de ensamble: le dices qué hacer con cada grupo de columnas y él lo aplica.

In [ ]:
deptos = pd.DataFrame({
    "m2":       [55, 80, 42, 120, 68, 50],
    "antiguedad": [5, 30, 12, 2, 18, 40],
    "estado":   ["Bueno", "Excelente", "Regular", "Excelente", "Bueno", "Regular"],
    "colonia":  ["Centro", "Roma", "Centro", "Polanco", "Roma", "Doctores"],
})

preparador = ColumnTransformer([
    ("numericas", StandardScaler(),                                   ["m2", "antiguedad"]),
    ("ordinal",   OrdinalEncoder(categories=[["Regular", "Bueno", "Excelente"]]), ["estado"]),
    ("nominal",   OneHotEncoder(handle_unknown="ignore", sparse_output=False),    ["colonia"]),
])
lista = preparador.fit_transform(deptos)
pd.DataFrame(lista, columns=preparador.get_feature_names_out()).round(2)

### 🔮 Ejercicio 10 — Cuenta las columnas
Antes de mirar la salida con detalle: si agregamos una columna `tiene_balcon` (sí/no) y la mandamos al grupo `nominal`, ¿cuántas columnas tendrá el resultado? Verifícalo ✏️ agregando `"tiene_balcon": ["sí", "no", "no", "sí", "sí", "no"]` a `deptos` y `"tiene_balcon"` a la lista del grupo `nominal`.

✍️ *Mi apuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Ahora hay 2 numéricas + 1 ordinal + 4 colonias = **7 columnas**. Con `tiene_balcon` se agregan 2 más (`tiene_balcon_no` y `tiene_balcon_sí`): **9**. (Con `drop="if_binary"` sería 8.)
</details>

---
## 7. 🎛️ Las perillas de la preparación

Los preparadores también tienen sus "hiperparámetros". Estos son los que más vas a usar:

| Herramienta · perilla | Qué controla | Opciones / efecto | Punto de partida |
|---|---|---|---|
| Escalador | Cómo se ponen las numéricas en escala común | `Standard` (general) · `MinMax` (rango 0–1) · `Robust` (con atípicos) | `StandardScaler` |
| `OneHotEncoder(handle_unknown)` | Qué hacer con una categoría nueva | `"error"` rompe · `"ignore"` pone todo en 0 | `"ignore"` |
| `OneHotEncoder(min_frequency)` | Agrupar categorías raras | Subirlo ⬆️ → menos columnas, se pierde detalle | `None`, o ~1 % de los datos si hay muchas categorías |
| `OneHotEncoder(drop)` | Quitar una columna redundante | `"if_binary"` → 1 columna para sí/no | `None` o `"if_binary"` |
| `OrdinalEncoder(categories)` | El orden de las categorías | Tú lo das; si no, es alfabético | Siempre explícito |
| `TargetEncoder(smooth)` | Cuánto se "acercan" las categorías raras al promedio general | Subirlo ⬆️ → más prudente · Bajarlo ⬇️ → confía más en cada categoría (riesgo de sobreajuste) | `"auto"`; un número (5–20) si hay categorías con 1–2 ejemplos |

### 🧭 Guía rápida: ¿qué codificación uso?
```
¿La categoría tiene orden natural?  ── Sí ──► Ordinal (con el orden explícito)
        │
        No
        ▼
¿Tiene pocas categorías (≲ 15)?     ── Sí ──► One-hot (handle_unknown="ignore")
        │
        No (cientos o miles)
        ▼
Target encoding (con suavizado) o one-hot agrupando las raras (min_frequency)
```

---
## 8. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué hacer |
|---|---|---|
| KNN o SVM funcionan mucho peor de lo esperado | Columnas sin escalar | Escalar (dentro de un pipeline) |
| Escalar a un árbol "no sirve de nada" | Es normal: los árboles no lo necesitan | No pierdas tiempo; tampoco hace daño |
| El modelo cree que "Oro < Plata" | Codificación ordinal con orden alfabético | Dar el orden con `categories=` |
| El sistema se cae con una categoría nueva | `handle_unknown="error"` (por defecto) | `handle_unknown="ignore"` |
| Miles de columnas tras el one-hot | Columna con muchas categorías (código postal, ID) | Agrupar raras, target encoding… o preguntarte si esa columna sirve |
| Resultados demasiado buenos con target encoding | Se calculó el promedio usando la etiqueta de la misma fila | `TargetEncoder` (ajuste cruzado), dentro de un pipeline |
| El escalador aprendió la media de **todos** los datos | Se ajustó antes de dividir en entrenamiento y prueba | Fuga de datos: Lección 2.6 |

### 🔮 Ejercicio 11 — ¿Escalar al revés?
Un compañero dice: *"Como el árbol no necesita escalado, voy a escalar solo para el KNN y para el árbol uso los datos crudos"*. ¿Es correcto?

<details><summary><b>👀 Ver respuesta</b></summary>

**Sí, es correcto** y es una práctica normal: cada modelo con su propia preparación. Gracias a los pipelines (`make_pipeline(StandardScaler(), KNN)` y `make_pipeline(DecisionTree)`), cada uno lleva su preparación "pegada" y no se confunden. Lo que **no** debe hacer es ajustar el escalador con todos los datos antes de dividirlos (Lección 2.6).
</details>

---
## 9. 🧠 Resumen en 5 puntos
1. Preparar los datos es ponerlos en un **idioma común** de números que el modelo pueda comparar con justicia.
2. Los modelos que usan **distancias o sumas ponderadas** (KNN, SVM, lineales, redes) necesitan **escalar**; los **árboles** no.
3. `StandardScaler` es el punto de partida; `RobustScaler` si hay atípicos; `MinMaxScaler` si necesitas un rango 0–1.
4. Categorías: **ordinal** si hay orden (¡dalo tú!), **one-hot** si no lo hay y son pocas, **target encoding** (con suavizado) si son muchísimas.
5. `ColumnTransformer` aplica a cada columna su tratamiento; dentro de un **pipeline**, todo se aprende solo con el entrenamiento.

---
## 10. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 10.1 Repaso rápido (opción múltiple)

**1.** ¿Cuál de estos modelos **no** necesita escalado?
- a) KNN · b) SVM · c) Árbol de decisión · d) Red neuronal

**2.** Una columna `talla` tiene S, M, L y XL. ¿Qué codificación es más adecuada?
- a) One-hot · b) Ordinal con el orden S < M < L < XL · c) Ordinal alfabético · d) Ninguna

**3.** Una columna `pais` tiene 5 valores. Tras un one-hot, ¿cuántas columnas genera?
- a) 1 · b) 4 o 5 · c) 25 · d) Depende del número de filas

**4.** Tus datos tienen ingresos con algunos millonarios que son **reales**. ¿Qué escalador eliges para KNN?
- a) `MinMaxScaler` · b) `RobustScaler` · c) Ninguno · d) Borro a los millonarios

**5.** ¿Cuál es el principal riesgo del target encoding hecho "a mano" (promedio simple por categoría)?
- a) Crea demasiadas columnas · b) Las categorías con pocos ejemplos reciben valores extremos que "filtran" su propia etiqueta · c) No funciona con números · d) Es muy lento

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **c)** El árbol pregunta columna por columna; la escala no cambia el orden de los datos.
2. **b)** La talla tiene orden natural y conviene conservarlo. One-hot (a) también funcionaría, pero tira esa información.
3. **b)** 5 columnas (o 4 si quitas una con `drop="first"`, porque la quinta se deduce de las otras).
4. **b)** `RobustScaler` usa la mediana y el rango intercuartílico, que no se dejan arrastrar por los extremos.
5. **b)** Una categoría con un solo cliente recibe su propia etiqueta como característica: el modelo "ve la respuesta" y sobreajusta.
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** Codificas `ciudad` con one-hot y `talla` con ordinal. Un compañero propone hacerlo al revés: one-hot para `talla` y ordinal para `ciudad`. ¿Cuál de los dos cambios es más grave y por qué?

**B. Caso de análisis 💼** Una cadena de farmacias quiere predecir las ventas de cada sucursal. Tiene la columna `codigo_sucursal` con **1,200** sucursales distintas, algunas abiertas hace una semana. ¿Qué codificación propondrías? ¿Qué pasa con las sucursales nuevas?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** El más grave es **ordinal para `ciudad`**: inventa un orden falso (¿Guadalajara "entre" CDMX y Monterrey?) que el modelo usará como si fuera real. One-hot para `talla` solo **desperdicia** el orden; no inventa nada falso.

**B.** One-hot crearía 1,200 columnas, la mayoría con pocos datos. Buenas opciones: **target encoding con suavizado** (las sucursales nuevas, con pocos datos, quedarán cerca del promedio general, que es una estimación prudente) o agrupar las raras. Aún mejor, complementar con características que **describan** a la sucursal (tamaño, zona, si está en centro comercial): así una sucursal nueva no es "desconocida", se parece a otras.
</details>

### 10.3 🧩 Práctica integradora: "¿Quién contrata el seguro?"

Una aseguradora te da datos de 1,500 personas a las que les ofreció un seguro de gastos médicos y si lo **contrataron**. Usarás un KNN (que necesita buena preparación) para predecirlo.

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Mira los datos**

In [ ]:
rng = np.random.default_rng(11)
n = 1_500
seguro = pd.DataFrame({
    "edad":        rng.integers(18, 70, n),
    "ingreso":     rng.lognormal(10, 0.6, n).round(-2),
    "hijos":       rng.integers(0, 4, n),
    "escolaridad": rng.choice(["Secundaria", "Preparatoria", "Licenciatura", "Posgrado"], n),
    "region":      rng.choice(["Norte", "Centro", "Sur", "Occidente"], n),
    "canal":       rng.choice(["Teléfono", "Web", "Sucursal"], n),
})
nivel = seguro["escolaridad"].map({"Secundaria": 0, "Preparatoria": 1, "Licenciatura": 2, "Posgrado": 3})
puntaje = (0.04 * (seguro["edad"] - 40) + 0.8 * nivel + 0.5 * seguro["hijos"]
           + 1.2 * (seguro["canal"] == "Sucursal") - 0.8 * (seguro["region"] == "Sur") - 1.5)
seguro["contrato"] = (rng.random(n) < 1 / (1 + np.exp(-puntaje))).astype(int)

X_seg = seguro.drop(columns="contrato")
y_seg = seguro["contrato"]
print(f"Contrataron: {y_seg.mean():.0%}")
seguro.head()

**Paso 2 ✋ — Decide el tratamiento de cada columna.** Para `edad`, `ingreso`, `hijos`, `escolaridad`, `region` y `canal`: ¿escalar, ordinal o one-hot? ¿Qué escalador usarías para `ingreso`? (Pista: mira la columna en `seguro.describe()`.)

✍️ *Mis decisiones:*

<details><summary><b>👀 Ver respuesta</b></summary>

- `edad`, `hijos` → escalar (`StandardScaler`).
- `ingreso` → escalar; tiene una cola de ingresos muy altos, así que `RobustScaler` es una buena elección (`StandardScaler` también funcionaría).
- `escolaridad` → **ordinal** con el orden Secundaria < Preparatoria < Licenciatura < Posgrado.
- `region`, `canal` → **one-hot** (sin orden, pocas categorías).
</details>

**Paso 3 🔮 — Arma el preparador.** Antes de ejecutar: ¿cuántas columnas tendrá la tabla preparada?

✍️ *Mi apuesta:*

In [ ]:
preparador_seg = ColumnTransformer([
    ("num",     StandardScaler(), ["edad", "hijos"]),
    ("robusto", RobustScaler(),   ["ingreso"]),
    ("ord",     OrdinalEncoder(categories=[["Secundaria", "Preparatoria", "Licenciatura", "Posgrado"]]),
                ["escolaridad"]),
    ("onehot",  OneHotEncoder(handle_unknown="ignore"), ["region", "canal"]),
])
preparada = preparador_seg.fit_transform(X_seg)
print("Forma:", preparada.shape)
print(preparador_seg.get_feature_names_out())

<details><summary><b>👀 Ver respuesta</b></summary>

**11 columnas**: 2 escaladas + 1 robusta + 1 ordinal + 4 regiones + 3 canales.
</details>

**Paso 4 🔮 — ¿Cuánto vale la preparación?** Vamos a comparar tres versiones del mismo KNN con validación cruzada:
1. **Solo las numéricas, sin escalar** (lo que harías sin pensar: tirar las columnas de texto).
2. **Solo las numéricas, escaladas.**
3. **Todo preparado** (el preparador del paso 3).

Antes de ejecutar: ordénalas de peor a mejor.

✍️ *Mi apuesta:*

In [ ]:
numericas = ["edad", "ingreso", "hijos"]
versiones = {
    "1. numéricas sin escalar": (KNeighborsClassifier(n_neighbors=25), X_seg[numericas]),
    "2. numéricas escaladas  ": (make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=25)), X_seg[numericas]),
    "3. todo preparado       ": (make_pipeline(preparador_seg, KNeighborsClassifier(n_neighbors=25)), X_seg),
}
for nombre, (modelo, datos) in versiones.items():
    print(f"{nombre}: {cross_val_score(modelo, datos, y_seg, cv=5).mean():.1%}")
print(f"\nReferencia 'decir siempre la clase más común': {max(y_seg.mean(), 1 - y_seg.mean()):.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

1 → 2 → 3. Sin escalar (≈64.5 %), el KNN solo "ve" el ingreso, que aquí **no influye**, y queda igual que adivinar siempre la clase más común (≈65 %): ¡no aprendió nada! Escalando las numéricas sube apenas (≈65.5 %). Con todo preparado llega a **≈72 %**: la escolaridad, el canal y la región, que eran texto, ahora aportan información.

Mismo modelo, mismos datos: **la preparación hizo toda la diferencia.**
</details>

**Paso 5 ✏️ — Un error a propósito.** Cambia, en el preparador del paso 3, el `OrdinalEncoder` de escolaridad por uno **sin orden explícito** (borra `categories=[...]`, que queda alfabético: Licenciatura < Posgrado < Preparatoria < Secundaria). Vuelve a ejecutar los pasos 3 y 4. ¿Cambia la exactitud? ¿Por qué?

<details><summary><b>👀 Ver respuesta</b></summary>

**Casi nada** (≈72.4 % → ≈72.2 %). ¿Entonces el orden da igual? **No.** Aquí la escolaridad es solo una de 11 columnas y el KNN promedia 25 vecinos, así que el daño se diluye. Pero el orden alfabético dice que "Secundaria" es el nivel **más alto**: cualquier persona que lea o use ese modelo (o un modelo más sensible al orden) heredará una mentira.

Lección doble: **el orden de las categorías lo das tú**, y **que una métrica casi no cambie no demuestra que una decisión sea correcta**.
</details>

---
## 📚 Para profundizar
- 📘 **Documentación:** scikit-learn, guía de usuario, sección *Preprocessing data* (escaladores y codificadores) y *Target Encoder*.
- 📺 **Video:** StatQuest, "One-Hot, Label, Target and K-Fold Target Encoding, Clearly Explained" (inglés, con subtítulos).
- 📘 **Documentación:** scikit-learn, *ColumnTransformer for heterogeneous data*.

---
⬅️ **Anterior:** [2.2 Calidad de datos: faltantes, atípicos, duplicados y ruido](2.2-calidad-de-datos.ipynb) · ➡️ **Siguiente:** [2.4 Ingeniería de características: crear información útil](2.4-ingenieria-de-caracteristicas.ipynb)